In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Master 3-Tier Hierarchical Triage Benchmark Pipeline (`models/combined_hierarchical_triage_pipeline.ipynb`)

This benchmark notebook evaluates the end-to-end performance of the **3-Tier 4-Model Hierarchical LightGBM Triage Architecture** on the 1% holdout test set using the uniform 38-feature input matrix across all layers.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Environment, Configuration & Trained Sub-Models
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
l1_obj  <- readRDS(file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
l2_obj  <- readRDS(file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
l3a_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi23_model.rds"))
l3b_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi45_model.rds"))
lgb_l1_model  <- l1_obj$model
lgb_l2_model  <- l2_obj$model
lgb_l3a_model <- l3a_obj$model
lgb_l3b_model <- l3b_obj$model
scaler        <- l1_obj$scaler
# Load Master Dataset & Build Uniform 38-Feature Matrix
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
hr_rng   <- pulse_max - pulse_min
sbp_rng  <- sbp_max - sbp_min
rr_rng   <- resp_max - resp_min
spo2_rng <- spo2_max - spo2_min
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_min               = pulse_min,
  resp_min                = resp_min,
  spo2_min                = spo2_min,
  sbp_min                 = sbp_min,
  pulse_max               = pulse_max,
  resp_max                = resp_max,
  spo2_max                = spo2_max,
  sbp_max                 = sbp_max,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  hr_range                = hr_rng,
  rr_range                = rr_rng,
  spo2_range              = spo2_rng,
  sbp_range               = sbp_rng,
  shock_index             = t_hr / ifelse(t_sbp == 0, 1, t_sbp),
  hr_mid_to_triage        = t_hr - hr_rng,
  sbp_mid_to_triage       = t_sbp - sbp_rng,
  rr_mid_to_triage        = t_rr - rr_rng,
  spo2_mid_to_triage      = t_o2 - spo2_rng,
  rox_index               = t_o2 / ifelse(t_rr == 0, 1, t_rr),
  spo2_drop_ratio         = spo2_rng / ifelse(spo2_max == 0, 1, spo2_max),
  hr_instability_ratio    = hr_rng / (t_hr + 1),
  bif                     = (t_rr / ifelse(t_o2 == 0, 1, t_o2)) * 100
)
layer_feat_names <- names(df_master)
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}
in_train_val <- stratified_partition(df_master$target_col, p = 1 - config$training$test_size, seed = config$training$random_state)
test_df      <- df_master[-in_train_val, ]
test_scaled  <- apply_scaler(test_df, scaler)
cat(sprintf("Holdout Test Set Prepared with 38 Uniform Features (%d rows)\n", nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Evaluate Soft Probabilistic Joint Pipeline
# ---------------------------------------------------------
X_test <- as.matrix(test_scaled[, layer_feat_names])
p1_test  <- predict(lgb_l1_model,  X_test)
p2_test  <- predict(lgb_l2_model,  X_test)
p3a_test <- predict(lgb_l3a_model, X_test)
p3b_test <- predict(lgb_l3b_model, X_test)
probs_comb <- matrix(0, nrow = nrow(test_df), ncol = 5)
colnames(probs_comb) <- c("1", "2", "3", "4", "5")
probs_comb[, 1] <- p1_test
probs_comb[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_comb[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_comb[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_comb[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
preds_comb <- factor(apply(probs_comb, 1, which.max), levels = 1:5)
act_comb   <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)
compute_cm_metrics <- function(preds, refs) {
  tbl <- table(Prediction = preds, Reference = refs)
  lvls <- levels(refs)
  sens_v <- numeric(length(lvls))
  spec_v <- numeric(length(lvls))
  for (i in seq_along(lvls)) {
    l <- lvls[i]
    tp <- ifelse(l %in% rownames(tbl) && l %in% colnames(tbl), tbl[l, l], 0)
    fn <- sum(tbl[, l]) - tp
    fp <- sum(tbl[l, ]) - tp
    tn <- sum(tbl) - (tp + fn + fp)
    sens_v[i] <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
    spec_v[i] <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
  }
  bal_v <- (sens_v + spec_v) / 2
  return(list(table = tbl, Sensitivity = sens_v, Specificity = spec_v, BalancedAccuracy = bal_v))
}
cm_comb <- compute_cm_metrics(preds_comb, act_comb)
macro_bal <- mean(cm_comb$BalancedAccuracy)
cat("============================================================\n")
cat("   COMBINED 5-CLASS SOFT PIPELINE TEST REPORT (38 FEATS)\n")
cat("============================================================\n")
cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal))
cat("============================================================\n\n")
print(cm_comb$table)